# Foundry IQ knowledge base with Fabric Data Agent integration

This notebook creates a Foundry IQ knowledge base that combines indexed company documents with a **Fabric Data Agent Knowledge Source**.

- **Fabric Data Agent Knowledge Source**: connects Azure AI Search to a published data agent in a Fabric workspace for structured and analytical questions.
- **Delegated token**: uses the signed-in user's identity when Azure AI Search queries the protected Fabric source.

The Fabric Data Agent knowledge source is currently exposed by the preview Azure AI Search SDK even though its public documentation is limited.

## Step 1: Set up Azure AI Search, Azure OpenAI, and Fabric configuration

Load the resource configuration from the project `.env` file. This notebook requires the ID of the published Fabric Data Agent created during provisioning.

In [1]:
import json
import os

from azure.identity import AzureDeveloperCliCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
FABRIC_TENANT_ID = os.environ["FABRIC_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_DATA_AGENT_ID = os.environ["FABRIC_DATA_AGENT_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"
HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
FABRIC_DATA_AGENT_KNOWLEDGE_SOURCE_NAME = "fabric-data-agent-knowledge-source"
KNOWLEDGE_BASE_NAME = "multisource-fabric-data-agent-knowledge-base"

search_credential = AzureDeveloperCliCredential(tenant_id=AZURE_TENANT_ID)
print("Environment variables loaded")

Environment variables loaded


## Step 2: Authenticate with a delegated user token

The retrieval call needs an OAuth token for a signed-in user who can access the Fabric workspace and published data agent. Azure AI Search forwards this delegated identity when querying the protected source.

Run the cell below to sign in and request the Azure AI Search scope used by `query_source_authorization`.

In [2]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(tenant_id=FABRIC_TENANT_ID)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

/Users/pamelafox/iqdeepdive-foundryiq-1/.venv/lib/python3.14/site-packages/msal/oauth2cli/oauth2.py:453: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(


Acquired token for logged-in user


## Step 3: Create the search index knowledge sources

Create or update the same HR and health document sources used by the other multi-source labs.

- `hrdocs-knowledge-source`: points to the `hrdocs` search index
- `healthdocs-knowledge-source`: points to the `healthdocs` search index

In [3]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

index_client = SearchIndexClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    credential=search_credential,
    api_version=AZURE_SEARCH_API_VERSION,
)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Contoso HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Contoso health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

print("Search index knowledge sources created")

Search index knowledge sources created


## Step 4: Create the Fabric Data Agent knowledge source

The `azure-search-documents` package exposes a Fabric Data Agent source that identifies the published agent by its Fabric workspace ID and data agent ID.

In [4]:
from azure.search.documents.indexes.models import (
    FabricDataAgentKnowledgeSource,
    FabricDataAgentKnowledgeSourceParameters,
)

fabric_data_agent_knowledge_source = FabricDataAgentKnowledgeSource(
    name=FABRIC_DATA_AGENT_KNOWLEDGE_SOURCE_NAME,
    description="Data agent with access to product inventory, reviews, and website analytics",
    fabric_data_agent_parameters=FabricDataAgentKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        data_agent_id=FABRIC_DATA_AGENT_ID,
    ),
)
index_client.create_or_update_knowledge_source(
    knowledge_source=fabric_data_agent_knowledge_source
)

print("Fabric Data Agent knowledge source created")

Fabric Data Agent knowledge source created


## Step 5: Create the multi-source knowledge base

Combine the two document indexes and the Fabric Data Agent with an Azure OpenAI chat model. Answer synthesis lets the knowledge base produce one response across the selected sources, while low reasoning effort enables source selection and query planning.

In [5]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

knowledge_source_names = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    FABRIC_DATA_AGENT_KNOWLEDGE_SOURCE_NAME,
 ]

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base combining company documents, product data, website analytics",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in knowledge_source_names],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions=(
        "Use the Fabric Data Agent for structured product, inventory, and analytical questions. "
        "Use the search indexes for HR and health policy documents."
    ),
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")

Knowledge base created


## Step 6: Query the knowledge base

Ask a question that requires current inventory analysis from the Fabric Data Agent and role information from the indexed HR documents. The data agent is always queried for this request; agentic retrieval can decide whether each document source is also needed.

Retrieval can take longer than an index-only query because Azure AI Search invokes the published Fabric Data Agent.

In [6]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricDataAgentKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=KNOWLEDGE_BASE_NAME,
    credential=search_credential,
    api_version=AZURE_SEARCH_API_VERSION,
)

question = (
    "I'm a Contoso buyer preparing for the summer sale. "
    "Which products or product categories have the lowest or zero inventory right now, and where? "
    "Which Contoso job roles are responsible for inventory strategy and budget oversight?"
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        FabricDataAgentKnowledgeSourceParams(
            knowledge_source_name=FABRIC_DATA_AGENT_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=180,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)
display(Markdown(result.response[0].content[0].text))

Here’s a focused view of the most urgent summer-sale inventory gaps and the Contoso roles most closely tied to inventory strategy and budget oversight.

## Lowest or zero inventory right now, and where

There are **379 product-store records at or below reorder point** right now, including many with **zero available inventory**.[ref_id:1]

The most urgent shortfalls identified are:

- **Power Tools**: Heavy Duty Jigsaw at **Redmond**; Cordless Miter Saw at **Tacoma**; Compound Miter Saw at **Tacoma**; Hammer Drill at **Tacoma**; File Belt Sander at **Bellevue**; Sliding Miter Saw at **Bellevue**; Drywall Sander at **Redmond**.[ref_id:1]
- **Paint & Finishes**: Gloss Polyurethane at **Tacoma** and **Bellevue**; Roller Frame 9-inch at **Redmond** and **Bellevue**; Deck and Fence Stain at **Bellevue**.[ref_id:1]
- **Lumber & Building Materials**: PVC Trim Board at **Redmond**; WonderBoard Lite at **Seattle**; Douglas Fir 2x6x10 at **Tacoma**.[ref_id:1]
- **Plumbing**: Fiberglass Pipe Insulation at **Everett**; Kitchen Sink Faucet at **Everett**; Compression Fittings at **Everett**; Pre-Insulated Copper Pipe at **Everett**; Pipe Dope Stick at **Spokane**; Ball Valve at **Spokane**.[ref_id:1]
- **Electrical**: Track Lighting Kit at **Redmond**; Ceiling Box with Bracket at **Redmond**; several other electrical products also have **no available units** across **Tacoma, Bellevue, Seattle, and Kirkland**.[ref_id:1]
- **Garden & Outdoor**: LED Flood Light 30W at **Seattle**; Soaker Hose 25-foot at **Spokane**; All-Purpose Fertilizer 50lb at **Bellevue**.[ref_id:1]
- **Storage & Organization**: Chemical Storage Cabinet at **Redmond**; Modular Cabinet System at **Tacoma**.[ref_id:1]
- **Hardware and Hand Tools**: Security Hasp at **Spokane** and **Kirkland**; screwdriver and wrench sets at **Spokane** and **Bellevue**.[ref_id:1]

The shortage pattern is concentrated in **Bellevue, Redmond, Tacoma, Seattle, Kirkland, and Everett**, with additional zero-availability items in **Spokane**.[ref_id:1]

In many of these cases, **all on-hand units are reserved**, which leaves **zero saleable quantity** even when small positive on-hand stock still exists.[ref_id:1]

## Contoso job roles responsible for inventory strategy and budget oversight

Based on the role documents retrieved, the roles most directly aligned are:

- **Director of Operations**: responsible for developing and implementing strategies to **manage inventory and stock levels** and for managing the company **budget and financial operations**.[ref_id:0]
- **Vice President of Operations**: responsible for developing and implementing operational strategies and for overseeing **operational plans and budgets**.[ref_id:13]
- **Senior Manager of Operations**: responsible for overseeing operations including **supply chain, logistics, and finance**, and for developing, implementing, and monitoring **operational plans and budgets**.[ref_id:9][ref_id:18]
- **Manager of Operations**: responsible for overseeing day-to-day operations and for developing and managing **operational budgets**.[ref_id:16]
- **Chief Operating Officer (COO)**: responsible for organization-wide operational strategy and for managing the budget to ensure the organization operates within its means.[ref_id:7]
- **Chief Financial Officer (CFO)**: responsible for financial operations including **budgeting, forecasting, and financial planning and analysis**, making this role central to **budget oversight**.[ref_id:3]

There are also product-management roles with budget authority that may influence assortment and planning, though the retrieved documents do **not explicitly tie them to inventory ownership**:

- **Director of Product Management**: responsible for product strategies and for developing and managing **product budgets**.[ref_id:11][ref_id:19]
- **Manager of Product Management**: responsible for product strategy execution, pricing/cost models, and managing the **product development budget**.[ref_id:14]
- **Senior Manager of Product Management**: responsible for product strategies and for developing and managing **product budgets**.[ref_id:17]
- **Vice President of Product Management**: responsible for overall product strategy and roadmap leadership.[ref_id:12][ref_id:4]

## Practical takeaway for your summer sale prep

If your goal is to prioritize immediate replenishment risk, the biggest watchlist is in **Power Tools, Paint & Finishes, Plumbing, Electrical, and Lumber & Building Materials**, especially in **Bellevue, Redmond, Tacoma, Seattle, Everett, Kirkland, and Spokane**.[ref_id:1]

If you want, I can next turn this into a **store-by-store replenishment priority list** or a **category escalation summary for the operations and finance owners**.[ref_id:1][ref_id:0][ref_id:3]

## Step 7: Inspect retrieval activity

Review the query plan and source operations. A successful mixed-source request should include a Fabric Data Agent activity record and any search-index queries selected by agentic retrieval.

In [7]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

[
  {
    "type": "modelQueryPlanning",
    "id": 0,
    "inputTokens": 1540,
    "outputTokens": 117,
    "modelName": "gpt-5.4",
    "elapsedMs": 1637
  },
  {
    "type": "searchIndex",
    "id": 2,
    "knowledgeSourceName": "hrdocs-knowledge-source",
    "queryTime": "2026-10-14T18:12:51.2194844Z",
    "count": 36,
    "elapsedMs": 0,
    "searchIndexArguments": {
      "search": "inventory strategy budget oversight job roles",
      "filter": null,
      "semanticConfigurationName": "semantic-configuration",
      "sourceDataFields": [
        {
          "name": "snippet"
        },
        {
          "name": "uid"
        },
        {
          "name": "snippet_parent_id"
        },
        {
          "name": "blob_path"
        }
      ],
      "searchFields": [
        {
          "name": "snippet"
        }
      ]
    }
  },
  {
    "type": "searchIndex",
    "id": 3,
    "knowledgeSourceName": "hrdocs-knowledge-source",
    "queryTime": "2026-10-14T18:12:51.4551721Z",
  

## Step 8: Inspect search and Data Agent references

Print each reference as returned by the preview API. Compare the document citations with the Fabric Data Agent reference, synthesized response, and structured source data.

Do not assume the Data Agent payload uses the same fields as a `fabricOntology` reference; inspect its actual `type` and `sourceData` values.

In [8]:
references_json = [reference.as_dict() for reference in result.references or []]
print(json.dumps(references_json, indent=2))

[
  {
    "type": "searchIndex",
    "id": "0",
    "activitySource": 2,
    "rerankerScore": 3.4986732,
    "docKey": "810256edf976_aHR0cHM6Ly9tYWdvdHRlaWFkbHNnZW4yLmJsb2IuY29yZS53aW5kb3dzLm5ldC9sYWJkb2NzL3JvbGVfbGlicmFyeS5wZGY1_pages_14",
    "sourceData": {
      "uid": "810256edf976_aHR0cHM6Ly9tYWdvdHRlaWFkbHNnZW4yLmJsb2IuY29yZS53aW5kb3dzLm5ldC9sYWJkb2NzL3JvbGVfbGlicmFyeS5wZGY1_pages_14",
      "snippet_parent_id": "aHR0cHM6Ly9tYWdvdHRlaWFkbHNnZW4yLmJsb2IuY29yZS53aW5kb3dzLm5ldC9sYWJkb2NzL3JvbGVfbGlicmFyeS5wZGY1",
      "blob_path": "/hrdocs/role_library.pdf",
      "snippet": "written and verbal communication skills \n\n\u2022 Ability to manage and motivate a team \n\n\u2022 Analytical and problem-solving skills \n\n\u2022 Experience in budget management and financial analysis \n\n\u2022 Knowledge of SEO and web analytics tools \n\n\u2022 Ability to work independently and collaboratively in a fast-paced environment \n\nDirector of Operations \n\nJob Title: Director of Operations, C